# Linguistic features: syntax and paired comparisons

Achraf’s Day 2 analysis, based on Roland’s dependency parsing and Flo’s stylometric features. This notebook describes whole dialogues, not syntactic accommodation. Run the preprocessing and stylometric scripts first. See `docs/achraf/README.md`.

In [ ]:
from pathlib import Path
import hashlib
import json
import importlib.metadata
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() or (p / 'scripts/achraf').is_dir()), Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data/processed/achraf'
OUTPUT_DIR = PROJECT_ROOT / 'results/achraf/day2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECOMPUTE_SYNTAX = True
INPUT_FILE = DATA_DIR / 'model_inputs.csv'
STYLE_FILE = OUTPUT_DIR / 'stylometric_features.csv'
ROBUSTNESS_FILE = OUTPUT_DIR / 'repetition_robustness_comparable.csv'

def file_fingerprint(path):
    return hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()

METADATA_FILE = OUTPUT_DIR / 'day2_run_metadata.json'
CACHE_FILE = OUTPUT_DIR / 'day2_syntactic_features.csv'
previous_metadata = json.loads(METADATA_FILE.read_text(encoding='utf-8')) if METADATA_FILE.is_file() else {}

def validate_cache():
    if previous_metadata.get('cache_input_sha256_lf') != file_fingerprint(INPUT_FILE) or previous_metadata.get('cache_sha256_lf') != file_fingerprint(CACHE_FILE):
        raise ValueError('Cache or input changed. Rerun model inference.')


## 1. Validate the input

One human and one GPT row per pair. Cache mode (`RECOMPUTE_SYNTAX = False`) reuses an existing `day2_syntactic_features.csv`. It checks IDs and values but does not rerun the parser. Use only the supplied matching cache or recompute after changing input text. For Colab, explicitly set `DATA_DIR` and `OUTPUT_DIR` in the configuration cell.

In [ ]:
syntax_df = pd.read_csv(INPUT_FILE)
required = {'pair_id', 'context', 'source', 'text'}
if not required.issubset(syntax_df.columns):
    raise ValueError(f'Missing input columns: {sorted(required - set(syntax_df.columns))}')
if syntax_df.empty or syntax_df[list(required - {'review_notes'})].isna().any().any() or syntax_df['text'].str.strip().eq('').any():
    raise ValueError('Empty dataset, text or missing metadata')
if syntax_df.duplicated(['pair_id', 'source']).any() or set(syntax_df['source']) != {'human', 'gpt'}:
    raise ValueError('Invalid source labels or duplicate pair/source keys')
composition = syntax_df.groupby('pair_id')['source'].agg(set)
if not composition.map(lambda labels: labels == {'human', 'gpt'}).all() or syntax_df.groupby('pair_id')['context'].nunique().gt(1).any():
    raise ValueError('Incomplete pairs or inconsistent contexts')
print('Dialogues:', len(syntax_df), 'Matched pairs:', len(composition))


## 2. Dependency features

The root has depth 1. Tree traversal excludes punctuation and spaces. Dependency distance counts positions among non-punctuation, non-space tokens and excludes self-headed roots. Within-dialogue depth SD uses `ddof=0`. Maximum values can increase with dialogue length.

In [ ]:
def get_tree_depth(token):
    children = [child for child in token.children if not child.is_punct and (not child.is_space)]
    return 1 + max((get_tree_depth(child) for child in children), default=0)

def extract_dependency_features(text):
    doc = text if isinstance(text, spacy.tokens.Doc) else nlp(text)
    depths = []
    distances = []
    for sentence in doc.sents:
        tokens = [token for token in sentence if not token.is_punct and (not token.is_space)]
        if not tokens:
            continue
        positions = {token.i: position for position, token in enumerate(tokens)}
        depths.append(get_tree_depth(sentence.root))
        for token in tokens:
            if token.head.i != token.i and token.head.i in positions:
                distances.append(abs(positions[token.i] - positions[token.head.i]))
    return {'avg_tree_depth': float(np.mean(depths)) if depths else np.nan, 'max_tree_depth': max(depths) if depths else np.nan, 'std_tree_depth': float(np.std(depths)) if depths else np.nan, 'avg_dep_distance': float(np.mean(distances)) if distances else np.nan, 'max_dep_distance': max(distances) if distances else np.nan}


In [ ]:
features = ['avg_tree_depth', 'max_tree_depth', 'std_tree_depth', 'avg_dep_distance', 'max_dep_distance']
if RECOMPUTE_SYNTAX:
    import spacy
    from tqdm.auto import tqdm
    nlp = spacy.load('en_core_web_sm')
    print('spaCy:', spacy.__version__, 'Model:', nlp.meta.get('version'))
    print('Example:', extract_dependency_features('The worried student reads a book.'))
    documents = nlp.pipe(syntax_df['text'].tolist(), batch_size=64, disable=['ner'])
    feature_rows = [extract_dependency_features(doc) for doc in tqdm(documents, total=len(syntax_df), desc='Extracting syntactic features')]
    feature_df = pd.DataFrame(feature_rows, index=syntax_df.index)
    syntax_results = pd.concat([syntax_df[['pair_id', 'context', 'source']], feature_df], axis=1)
else:
    validate_cache()
    syntax_results = pd.read_csv(OUTPUT_DIR / 'day2_syntactic_features.csv')
    print('Reusing supplied syntax scores. Parser inference was not rerun.')
expected_keys = set(zip(syntax_df['pair_id'], syntax_df['source']))
if syntax_results.duplicated(['pair_id', 'source']).any() or set(zip(syntax_results['pair_id'], syntax_results['source'])) != expected_keys:
    raise ValueError('Syntax cache/extraction IDs do not match the input')
if not np.isfinite(syntax_results[features].to_numpy()).all():
    raise ValueError('Undefined syntax features. Inspect these dialogues before comparing them.')
context_check = syntax_df[['pair_id', 'source', 'context']].merge(syntax_results[['pair_id', 'source', 'context']], on=['pair_id', 'source'], suffixes=('_input', '_syntax'), validate='one_to_one')
if not context_check['context_input'].equals(context_check['context_syntax']):
    raise ValueError('Syntax contexts differ from input')
print(syntax_results.groupby('source')[features].agg(['mean', 'median']).round(3).T)


## 3. Paired effects and syntax distributions

All differences are GPT minus human. Cohen’s dz divides the mean paired difference by its sample SD (`ddof=1`). Hiding boxplot outliers does not remove observations.

In [ ]:
paired_syntax = syntax_results.pivot(index='pair_id', columns='source', values=features)
comparison_rows = []
for feature in features:
    human = paired_syntax[feature, 'human']
    gpt = paired_syntax[feature, 'gpt']
    difference = gpt - human
    difference_sd = difference.std(ddof=1)
    comparison_rows.append({'feature': feature, 'n_pairs': difference.count(), 'human_mean': human.mean(), 'gpt_mean': gpt.mean(), 'mean_difference': difference.mean(), 'cohens_dz': difference.mean() / difference_sd if difference_sd > 0 else np.nan, 'gpt_higher_pct': difference.gt(0).mean() * 100})
syntax_comparison = pd.DataFrame(comparison_rows)
print(syntax_comparison.round(3).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
plot_features = [('avg_tree_depth', 'Average dependency-tree depth', 'Levels'), ('avg_dep_distance', 'Average dependency distance', 'Token positions'), ('std_tree_depth', 'Within-dialogue tree-depth variability', 'Standard deviation of depth')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (feature, title, unit) in zip(axes, plot_features):
    human = syntax_results.loc[syntax_results['source'] == 'human', feature]
    gpt = syntax_results.loc[syntax_results['source'] == 'gpt', feature]
    ax.boxplot([human, gpt], tick_labels=['Human', 'GPT'], showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(unit)
fig.suptitle(f'Syntactic features — {syntax_df.pair_id.nunique():,} matched pairs')
fig.text(0.5, 0.01, 'Each observation represents one dialogue. Outlier points are hidden; all observations are retained.', ha='center', fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 0.94])
fig.savefig(OUTPUT_DIR / 'day2_syntactic_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
syntax_comparison.to_csv(OUTPUT_DIR / 'day2_syntactic_comparison.csv', index=False)


## 4. Merge the matching stylistic table

Fail on duplicate keys, missing pairs or mismatched IDs. No inner join should silently reduce the dataset. The numeric-only human dialogue has undefined word length, contraction and bigram rates. Historical question-mark frequency remains zero for this row.

In [ ]:
style_df = pd.read_csv(STYLE_FILE)
style_keys = set(zip(style_df['pair_id'], style_df['source']))
if style_df.duplicated(['pair_id', 'source']).any() or style_keys != expected_keys:
    raise ValueError('Stylistic and syntactic keys differ or contain duplicates')
combined_features = style_df.merge(syntax_results, on=['pair_id', 'source'], how='inner', validate='one_to_one')
if len(combined_features) != len(syntax_df):
    raise ValueError('Merge lost dialogue rows')
print('Combined shape:', combined_features.shape)


In [ ]:
style_plots = [('Longueur moyenne des mots', 'Average word length', 'Characters', 1), ('Contraction frequency', 'Contraction frequency', 'Per 100 words', 1), ("Ponctuation '?'", 'Question-mark frequency', 'Per 100 words', 1), ('Bigram repetition rate', 'Raw bigram repetition', 'Percent', 100)]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
style_summary_rows = []
for ax, (feature, title, unit, scale) in zip(axes.flat, style_plots):
    paired = combined_features.pivot(index='pair_id', columns='source', values=feature).dropna(subset=['human', 'gpt'])
    human = paired['human'] * scale
    gpt = paired['gpt'] * scale
    difference = gpt - human
    difference_sd = difference.std(ddof=1)
    ax.boxplot([human, gpt], tick_labels=['Human', 'GPT'], showfliers=False)
    ax.set_title(f'{title} — {len(paired):,} pairs')
    ax.set_ylabel(unit)
    style_summary_rows.append({'feature': feature, 'unit': unit, 'n_pairs': len(paired), 'human_mean': human.mean(), 'gpt_mean': gpt.mean(), 'mean_difference': difference.mean(), 'cohens_dz': difference.mean() / difference_sd if difference_sd > 0 else np.nan})
style_comparison = pd.DataFrame(style_summary_rows)
fig.suptitle('Lexical and writing-style features')
fig.text(0.5, 0.01, 'Outlier points are hidden; calculations retain all valid pairs. Raw repetition is sensitive to text length.', ha='center', fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
fig.savefig(OUTPUT_DIR / 'day2_stylometric_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
style_comparison.to_csv(OUTPUT_DIR / 'day2_stylometric_comparison.csv', index=False)
print(style_comparison.round(3).to_string(index=False))


## 5. Repetition length sensitivity

Compare the same eligible IDs in both window conditions. Each dialogue score averages ten sampled contiguous bigram windows, seed 42. This diagnostic is different from applying a meaning-preserving transformation to evaluate classifier robustness.

In [ ]:
robustness_df = pd.read_csv(ROBUSTNESS_FILE)
robustness_plot = robustness_df.loc[robustness_df['aggregation'] == 'Mean'].set_index('sampling').reindex(['Pairwise minimum', 'Fixed 30'])
if robustness_plot.index.has_duplicates or robustness_plot.isna().any().any() or robustness_plot['n_pairs'].nunique() != 1:
    raise ValueError('Missing conditions or inconsistent comparison sample')
n_comparable = int(robustness_plot['n_pairs'].iloc[0])


In [ ]:
x = np.arange(len(robustness_plot))
fig, ax = plt.subplots(figsize=(8, 4.5))
human_bars = ax.bar(x - 0.18, robustness_plot['human_rate'], width=0.36, label='Human')
gpt_bars = ax.bar(x + 0.18, robustness_plot['gpt_rate'], width=0.36, label='GPT')
ax.bar_label(human_bars, fmt='%.2f', padding=3)
ax.bar_label(gpt_bars, fmt='%.2f', padding=3)
ax.set_xticks(x, ['Pairwise minimum', 'Fixed 30 bigrams'])
ax.set_ylabel('Mean repeated-bigram rate (%)')
ax.set_title(f'Length-controlled repetition — same {n_comparable:,} matched pairs')
ax.set_ylim(0, robustness_plot[['human_rate', 'gpt_rate']].max().max() * 1.25)
ax.legend()
fig.text(0.5, 0.01, 'Within each dialogue, rates are averaged over 10 sampled contiguous bigram windows.', ha='center', fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(OUTPUT_DIR / 'day2_repetition_length_control.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Save results and provenance

Inputs remain unchanged. No automatic upload, Git operation or Colab-only download runs in this notebook.

In [ ]:
syntax_results.to_csv(OUTPUT_DIR / 'day2_syntactic_features.csv', index=False)
syntax_comparison.to_csv(OUTPUT_DIR / 'day2_syntactic_comparison.csv', index=False)
style_comparison.to_csv(OUTPUT_DIR / 'day2_stylometric_comparison.csv', index=False)
metadata = {'input_sha256': hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest(), 'syntax_inference_rerun': RECOMPUTE_SYNTAX, 'parser_model': 'en_core_web_sm', 'n_pairs': int(syntax_df.pair_id.nunique())}
if RECOMPUTE_SYNTAX:
    metadata.update({'spacy_version': spacy.__version__, 'parser_model_version': nlp.meta.get('version')})
metadata['cache_input_sha256_lf'] = file_fingerprint(INPUT_FILE)
metadata['cache_sha256_lf'] = file_fingerprint(CACHE_FILE)
metadata['inference_provenance'] = {'python_version': sys.version, 'packages': {package: importlib.metadata.version(package) for package in ['numpy', 'pandas', "spacy"]}, 'model_revision': 'Not recorded'} if RECOMPUTE_SYNTAX else previous_metadata.get('inference_provenance', {'status': 'Historical inference details unavailable'})
(OUTPUT_DIR / 'day2_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved results to:', OUTPUT_DIR)
